In [8]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/rcrzlbrd/data-mining-lab/main/playstore_cleaned.csv"
df = pd.read_csv(url)

### Reto 1: La Trampa de las Unidades de Medida
#### El equipo de infraestructura quiere saber si las aplicaciones más pesadas tienen menos descargas. Para averiguarlo, necesitan que la columna Size sea completamente numérica. Sin embargo, si revisan la columna, encontrarán valores como "19M" (Megabytes), "201k" (Kilobytes) y un texto molesto que dice "Varies with device".

#### Tu misión algorítmica:
#### Escribe una función pura en Python que reciba un string. Si el string termina en 'M', quítale la 'M' y conviértelo a flotante (dejándolo como Megabytes). Si termina en 'k', quítale la 'k', conviértelo a flotante y divídelo entre 1024 (para pasarlo también a Megabytes). Si dice "Varies with device", devuélvelo como un valor nulo de Numpy (np.nan).
#### Aplica esta función a toda la columna utilizando el método .apply().
#### Pregunta a responder: Una vez convertida la columna a valores numéricos (Megabytes), ejecuta el método .mean(). ¿Cuál es el peso promedio en Megabytes de las apps en la Play Store?

In [9]:
def parse_size(value):
    if value == "Varies with device":
        return np.nan
    if isinstance(value, str):
        if value.endswith("M"):
            return float(value[:-1])
        elif value.endswith("k"):
            return float(value[:-1]) / 1024
    return np.nan


df["Size"] = df["Size"].apply(parse_size)

mean_size = df["Size"].mean()
print(mean_size)

20.41973193945847


### Reto 2: El Tipo de Dato Cronológico
#### Ningún análisis de software está completo sin analizar el tiempo. La columna Last Updated tiene fechas escritas como texto: "January 7, 2018". Para un modelo matemático o una serie de tiempo, eso es texto inservible.

#### Tu misión algorítmica:
#### Investiga y utiliza la función pd.to_datetime() de Pandas para sobrescribir la columna Last Updated, convirtiéndola del tipo string (Object) al tipo nativo datetime64.
#### Ahora que es un objeto de tiempo, Pandas te permite extraer componentes específicos. Crea una nueva columna llamada Year_Updated extrayendo únicamente el año (df['Last Updated'].dt.year).
#### Pregunta a responder: Utilizando la sumarización categórica (value_counts()) sobre tu nueva columna Year_Updated, ¿en qué año se actualizó la mayor cantidad de aplicaciones en nuestro dataset?

In [10]:
df["Last Updated"] = pd.to_datetime(df["Last Updated"])

df["Year_Updated"] = df["Last Updated"].dt.year

print(df["Year_Updated"].value_counts())


Year_Updated
2018    6271
2017    1785
2016     779
2015     448
2014     203
2013     108
2012      26
2011      15
2010       1
Name: count, dtype: int64


### Reto 3: La Decisión Arquitectónica
#### Al resolver el Reto 1, introdujiste intencionalmente valores NaN en las aplicaciones cuyo tamaño decía "Varies with device".

#### Tu misión algorítmica: Evalúa cuántos registros quedaron vacíos. Como ingenieros, decidan y apliquen la mejor técnica: ¿es matemáticamente más sano borrar esas filas con un .dropna() porque el peso de la app es crítico, o prefieren rellenar ese hueco imputando la mediana global del peso de las apps?
#### Pregunta a responder: Redacten una breve justificación técnica de 3 líneas explicando qué método eligieron y por qué lo consideran superior para no dañar al modelo de predicción.

In [ ]:
print(df['Size'].isnull().sum())

median_size = df["Size"].median()
df["Size"] = df["Size"].fillna(median_size)

1227


##### Es mejor optar por rellenar con la mediana es mejor a simplemente borrar las filas con .dropna() porque estas representan un porcentaje mayor al 10%, introduciendo un severo sesgo. La mediana preserva la masa muestral sin verse afectada por outliers extremos de apps de varios gigabytes, manteniendo estable la distribución de varianza para la fase de entrenamiento.